In [9]:
import torch
import os
import json
from pathlib import Path
from safetensors import safe_open
from transformers import AutoConfig, AutoModelForCausalLM
from transformers.modeling_utils import no_init_weights

CONFIG = {
    "num_layers": 80,
    "hidden_size": 8192,
    "num_attention_heads": 64,
    "num_key_value_heads": 8,
    "seq_length": 8192,
    "max_position_embeddings": 8192,
    "ffn_hidden_size": 28672,
    "tensor_model_parallel_size": 8,
    "pipeline_model_parallel_size": 5,
    "rotary_position_embeddings_theta": 500000,
    "attention_dropout": 0.0,
    "hidden_dropout": 0.0,
    "use_rotary_position_embeddings": True,
    "swiglu": True,
    "bf16": True,
}

In [10]:
def divide(numerator, denominator):
    """Ensure numerator is divisible by denominator."""
    assert numerator % denominator == 0
    return numerator // denominator


class LlamaConverter:
    def __init__(self):

        self.home = os.environ["HOME"]
        self.hf_path = os.path.join(self.home, "models/llama-3.1-70b")
        self.save_path = os.path.join(self.home, "models/llama-3.1-70b-megads")
        
        if not os.path.exists(self.hf_path):
            raise ValueError(f"Model path {self.hf_path} does not exist")
            
        # Load HF config and model
        print(f"Loading Llama model from {self.hf_path}")
        self.hf_config = AutoConfig.from_pretrained(
            self.hf_path, trust_remote_code=True
        )

        self.tp_size = CONFIG["tensor_model_parallel_size"]
        self.pp_size = CONFIG["pipeline_model_parallel_size"]

        # Initialize model without weights first
        with no_init_weights():
            self.hf_model = AutoModelForCausalLM.from_config(
                self.hf_config, trust_remote_code=True
            )

        # Load weights
        self.hf_weights = self._load_weights()

        # Initialize attributes
        self.hidden_size = CONFIG["hidden_size"]
        self.num_attention_heads = CONFIG["num_attention_heads"]
        self.num_key_value_heads = CONFIG["num_key_value_heads"]
        self.vocab_size = self.hf_config.vocab_size
        self.num_layers = CONFIG["num_layers"]

        # Megatron specific
        self.padded_vocab_size = divide(self.vocab_size, self.tp_size) * self.tp_size

        # Initialize output weights dict
        self.output_weights = {}

    def _load_weights(self):
        """Load HF weights using sharded loading."""
        weights = {}
        safetensor_files = list(Path(self.hf_path).glob("*.safetensors"))
        for shard_file in sorted(safetensor_files):
            with safe_open(shard_file, framework="pt", device="cpu") as f:
                for k in f.keys():
                    weights[k] = f.get_tensor(k)
        return weights

    def _split_tensor(self, tensor, dim, num_partitions):
        """Split a tensor into N chunks along specified dimension."""
        partition_size = divide(tensor.size(dim), num_partitions)
        return torch.split(tensor, partition_size, dim=dim)

    def _calculate_padded_vocab_size(self, vocab_size, tp_size):
        """Calculate padded vocab size that's divisible by tp_size."""
        return ((vocab_size + tp_size - 1) // tp_size) * tp_size

    def _validate_tensor_shapes(self):
        """Validate critical tensor shapes before conversion."""
        expected_hidden = self.hidden_size
        expected_heads = self.num_attention_heads

        for layer_idx in range(self.num_layers):
            prefix = f"model.layers.{layer_idx}"
            
            # Check attention shapes
            q = self.hf_weights[f"{prefix}.self_attn.q_proj.weight"]
            if q.size(1) != expected_hidden:
                raise ValueError(f"Unexpected hidden size in layer {layer_idx}")
                
            # Add more shape checks as needed

    def convert_embedding_weights(self):
        """Convert embedding and LM head weights."""
        emb = self.hf_weights["model.embed_tokens.weight"]
        # For Llama models, lm_head weight is tied to embeddings
        lm_head = self.hf_weights["model.embed_tokens.weight"]

        # Split embeddings for tensor parallelism
        split_emb = self._split_tensor(emb, 0, self.tp_size)
        split_lm_head = self._split_tensor(lm_head, 0, self.tp_size)

        # Add to output weights for first and last PP ranks
        for tp_rank in range(self.tp_size):
            # Embeddings go to first PP rank
            pp_key_first = f"pp_0_tp_{tp_rank}"
            if pp_key_first not in self.output_weights:
                self.output_weights[pp_key_first] = {}
            self.output_weights[pp_key_first]["word_embeddings.weight"] = split_emb[tp_rank]

            # LM head goes to last PP rank
            pp_key_last = f"pp_{self.pp_size-1}_tp_{tp_rank}"
            if pp_key_last not in self.output_weights:
                self.output_weights[pp_key_last] = {}
            self.output_weights[pp_key_last]["lm_head.weight"] = split_lm_head[tp_rank]

    def convert_transformer_weights(self):
        """Convert transformer layer weights."""
        layers_per_pp = divide(self.num_layers, self.pp_size)

        for layer_idx in range(self.num_layers):
            # Determine PP rank for this layer
            pp_rank = layer_idx // layers_per_pp
            local_layer_idx = layer_idx % layers_per_pp

            # Get layer prefix
            hf_prefix = f"model.layers.{layer_idx}"

            # Convert attention weights
            self._convert_attention_weights(local_layer_idx, hf_prefix, pp_rank)

            # Convert MLP weights
            self._convert_mlp_weights(local_layer_idx, hf_prefix, pp_rank)

            # Convert layer norms (no splitting needed)
            for norm_name in ["input_layernorm", "post_attention_layernorm"]:
                weight = self.hf_weights[f"{hf_prefix}.{norm_name}.weight"]
                for tp_rank in range(self.tp_size):
                    pp_key = f"pp_{pp_rank}_tp_{tp_rank}"
                    if pp_key not in self.output_weights:
                        self.output_weights[pp_key] = {}
                    self.output_weights[pp_key][
                        f"layers.{local_layer_idx}.{norm_name}.weight"
                    ] = weight

    def _convert_attention_weights(self, layer_idx, hf_prefix, pp_rank):
        """Convert attention weights for a layer."""
        # Get Q,K,V weights
        q = self.hf_weights[f"{hf_prefix}.self_attn.q_proj.weight"]
        k = self.hf_weights[f"{hf_prefix}.self_attn.k_proj.weight"]
        v = self.hf_weights[f"{hf_prefix}.self_attn.v_proj.weight"]

        # Split Q,K,V for tensor parallelism
        split_q = self._split_tensor(q, 0, self.tp_size)
        split_k = self._split_tensor(k, 0, self.tp_size)
        split_v = self._split_tensor(v, 0, self.tp_size)

        # Get output projection
        attn_out = self.hf_weights[f"{hf_prefix}.self_attn.o_proj.weight"]
        split_attn_out = self._split_tensor(attn_out, 1, self.tp_size)

        # Add to output weights
        for tp_rank in range(self.tp_size):
            pp_key = f"pp_{pp_rank}_tp_{tp_rank}"
            if pp_key not in self.output_weights:
                self.output_weights[pp_key] = {}

            qkv = torch.cat(
                [split_q[tp_rank], split_k[tp_rank], split_v[tp_rank]], dim=0
            )
            self.output_weights[pp_key][
                f"layers.{layer_idx}.attention.query_key_value.weight"
            ] = qkv
            self.output_weights[pp_key][
                f"layers.{layer_idx}.attention.dense.weight"
            ] = split_attn_out[tp_rank]

    def _convert_mlp_weights(self, layer_idx, hf_prefix, pp_rank):
        """Convert MLP weights for a layer."""
        # Get MLP weights
        gate = self.hf_weights[f"{hf_prefix}.mlp.gate_proj.weight"]
        up = self.hf_weights[f"{hf_prefix}.mlp.up_proj.weight"]
        down = self.hf_weights[f"{hf_prefix}.mlp.down_proj.weight"]

        # Split for tensor parallelism
        split_gate = self._split_tensor(gate, 0, self.tp_size)
        split_up = self._split_tensor(up, 0, self.tp_size)
        split_down = self._split_tensor(down, 1, self.tp_size)

        # Add to output weights
        for tp_rank in range(self.tp_size):
            pp_key = f"pp_{pp_rank}_tp_{tp_rank}"
            if pp_key not in self.output_weights:
                self.output_weights[pp_key] = {}

            self.output_weights[pp_key][
                f"layers.{layer_idx}.mlp.dense_h_to_4h.weight"
            ] = torch.cat([split_gate[tp_rank], split_up[tp_rank]], dim=0)
            self.output_weights[pp_key][
                f"layers.{layer_idx}.mlp.dense_4h_to_h.weight"
            ] = split_down[tp_rank]

    def save_weights(self):
        """Save converted weights in Megatron format."""
        save_path = Path(self.save_path)
        save_path.mkdir(parents=True, exist_ok=True)

        # Save weights for each PP and TP rank combination
        for pp_rank in range(self.pp_size):
            for tp_rank in range(self.tp_size):
                rank_path = save_path / f"pp_{pp_rank}_tp_{tp_rank}"
                rank_path.mkdir(exist_ok=True)

                pp_key = f"pp_{pp_rank}_tp_{tp_rank}"
                torch.save(self.output_weights[pp_key], rank_path / "model_weights.pt")

        # Save config
        config = {
            **CONFIG,
            "vocab_size": self.vocab_size,
            "padded_vocab_size": self.padded_vocab_size,
        }

        with open(save_path / "model_config.json", "w") as f:
            json.dump(config, f, indent=4)

        print(f"Saved converted weights to {save_path}")


In [11]:
converter = LlamaConverter()


Loading Llama model from /ccs/home/tijmen/models/llama-3.1-70b


In [12]:
converter.hf_config

LlamaConfig {
  "_name_or_path": "/ccs/home/tijmen/models/llama-3.1-70b",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 8192,
  "initializer_range": 0.02,
  "intermediate_size": 28672,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 64,
  "num_hidden_layers": 80,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_type": "llama3"
  },
  "rope_theta": 500000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.48.0",
  "use_cache": true,
  "vocab_size": 128256
}

In [13]:
converter.hf_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 8192)
    (layers): ModuleList(
      (0-79): 80 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=8192, out_features=8192, bias=False)
          (k_proj): Linear(in_features=8192, out_features=1024, bias=False)
          (v_proj): Linear(in_features=8192, out_features=1024, bias=False)
          (o_proj): Linear(in_features=8192, out_features=8192, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=8192, out_features=28672, bias=False)
          (up_proj): Linear(in_features=8192, out_features=28672, bias=False)
          (down_proj): Linear(in_features=28672, out_features=8192, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((8192,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((8192,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((8192,), eps=1e-05)
    (rotary_

In [ ]:
converter.hf_weights

In [15]:
converter.convert_embedding_weights()


In [16]:
converter.convert_transformer_weights()


In [17]:
converter.save_weights()


Saved converted weights to /ccs/home/tijmen/models/llama-3.1-70b-megads
